[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/21_Decompression_Models.ipynb)

# DiveLab

## Notebook 21 — Decompression Models as Dynamical Systems

**Guiding question:** How can tissue gas loading be represented as hidden dynamical states driven by pressure history?

This notebook introduces decompression modeling from a systems perspective.

> **Important:** this is an educational model for learning dynamical systems. It is not a decompression planner and must not be used to plan real dives.

## Learning objectives

By the end of this notebook, you will be able to:

- model a tissue compartment as a first-order system;
- convert half-time into a rate constant;
- explain exponential uptake and elimination;
- compare fast and slow compartments;
- simulate multiple compartments;
- integrate compartment states along a depth profile;
- write the model in state-space and transfer-function form;
- connect compartment models to hidden-state estimation and dive computers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. The compartment model

A simplified inert-gas compartment follows:

$$
\boxed{
\dot P_t = k(P_a-P_t)
}
$$

where:

- $P_t$ is the compartment inert-gas tension;
- $P_a$ is the driving inert-gas pressure;
- $k$ is the compartment rate constant.

In simple words:

> the compartment moves toward the driving pressure, but not instantly.

# 2. First-order response

For constant $P_a$, the solution is:

$$
\boxed{
P_t(t)
=
P_a+
\left(P_t(0)-P_a\right)e^{-kt}
}
$$

This is exactly the same exponential form we saw for first-order control systems.

# 3. Half-time

A compartment is often described by a half-time:

$$
T_{1/2}.
$$

The rate constant is:

$$
\boxed{
k=\frac{\ln 2}{T_{1/2}}
}
$$

After one half-time, half the original difference from equilibrium remains.

In [ ]:
def rate_constant(half_time_min):
    return np.log(2) / half_time_min

for T_half in [5, 10, 20, 40, 80]:
    print(f"{T_half:3d} min -> k = {rate_constant(T_half):.5f} 1/min")

# 4. Fast and slow compartments

Short half-time:

- large $k$;
- fast response;
- stronger sensitivity to recent changes.

Long half-time:

- small $k$;
- slow response;
- longer memory.

In [ ]:
half_times = [5, 20, 80]
t = np.linspace(0, 120, 1000)

P0_tissue = 0.79
P_drive = 2.37

for T_half in half_times:
    k = rate_constant(T_half)
    Pt = P_drive + (P0_tissue - P_drive) * np.exp(-k*t)
    plt.plot(t, Pt, label=f"{T_half} min")

plt.xlabel("Time [min]")
plt.ylabel("Compartment tension [simplified pressure units]")
plt.title("Fast and slow compartment uptake")
plt.grid(True)
plt.legend()
plt.show()

A multi-compartment model is therefore a collection of memories operating at different time scales.

# 5. Verify the half-time rule

In [ ]:
T_half = 20.0
k = rate_constant(T_half)

fraction_remaining = np.exp(-k*T_half)
print("Fraction remaining after one half-time:", fraction_remaining)

The result is:

$$
0.5.
$$

After two half-times, $0.25$ remains; after three, $0.125$.

# 6. Uptake and elimination

The same equation describes both.

If:

$$
P_a>P_t,
$$

then:

$$
\dot P_t>0
$$

and the compartment loads gas.

If:

$$
P_a<P_t,
$$

then:

$$
\dot P_t<0
$$

and the compartment eliminates gas.

In [ ]:
t = np.linspace(0, 60, 600)

T_half = 20
k = rate_constant(T_half)

Pt_up = 2.4 + (0.8 - 2.4)*np.exp(-k*t)
Pt_down = 0.8 + (2.4 - 0.8)*np.exp(-k*t)

plt.plot(t, Pt_up, label="Uptake")
plt.plot(t, Pt_down, label="Elimination")
plt.xlabel("Time [min]")
plt.ylabel("Compartment tension")
plt.title("Uptake and elimination")
plt.grid(True)
plt.legend()
plt.show()

# 7. Driving pressure from depth

For teaching purposes, use:

$$
P_{\text{abs}}(z)\approx 1+\frac{z}{10}
$$

in bar.

For inert-gas fraction $f_I$:

$$
\boxed{
P_a(z)=f_I P_{\text{abs}}(z)
}
$$

This is a simplified driving-pressure model.

In [ ]:
def ambient_pressure_bar(depth_m):
    return 1.0 + depth_m/10.0

def inert_gas_pressure(depth_m, inert_fraction=0.79):
    return inert_fraction * ambient_pressure_bar(depth_m)

# 8. Constant-depth exposure

Start from surface equilibrium and move to a constant depth.

In [ ]:
depth = 20.0
Pa = inert_gas_pressure(depth)
Pt0 = inert_gas_pressure(0)

t = np.linspace(0, 120, 1000)

for T_half in [5, 20, 80]:
    k = rate_constant(T_half)
    Pt = Pa + (Pt0-Pa)*np.exp(-k*t)
    plt.plot(t, Pt, label=f"{T_half} min")

plt.axhline(Pa, linestyle="--", label="Driving pressure")
plt.xlabel("Time [min]")
plt.ylabel("Compartment inert-gas tension [bar, simplified]")
plt.title("Constant-depth uptake")
plt.grid(True)
plt.legend()
plt.show()

# 9. Changing depth requires numerical integration

For a changing profile:

$$
\dot P_t
=
k(P_a(t)-P_t).
$$

Euler integration gives:

$$
P_{t,k+1}
=
P_{t,k}
+
k(P_{a,k}-P_{t,k})\Delta t.
$$

In [ ]:
def simulate_compartment(
    depth_fn,
    half_time_min,
    duration_min,
    dt_min=0.01,
    inert_fraction=0.79
):
    t = np.arange(0, duration_min + dt_min, dt_min)
    depth = np.zeros_like(t)
    Pa = np.zeros_like(t)
    Pt = np.zeros_like(t)

    Pt[0] = inert_gas_pressure(0, inert_fraction)
    k = rate_constant(half_time_min)

    for i in range(len(t)-1):
        depth[i] = depth_fn(t[i])
        Pa[i] = inert_gas_pressure(depth[i], inert_fraction)

        Pt[i+1] = Pt[i] + k*(Pa[i]-Pt[i])*dt_min

    depth[-1] = depth_fn(t[-1])
    Pa[-1] = inert_gas_pressure(depth[-1], inert_fraction)

    return t, depth, Pa, Pt

# 10. Example depth profile

In [ ]:
def profile(t_min):
    if t_min < 3:
        return 20/3*t_min
    if t_min < 23:
        return 20.0
    if t_min < 27:
        return 20 - 5*(t_min-23)
    return 0.0

tt = np.linspace(0, 50, 1000)
zz = np.array([profile(x) for x in tt])

plt.plot(tt, zz)
plt.gca().invert_yaxis()
plt.xlabel("Time [min]")
plt.ylabel("Depth [m]")
plt.title("Example profile")
plt.grid(True)
plt.show()

# 11. One compartment along the profile

In [ ]:
t_c, z_c, Pa_c, Pt_c = simulate_compartment(
    profile,
    half_time_min=20,
    duration_min=50
)

plt.plot(t_c, Pa_c, label="Driving inert-gas pressure")
plt.plot(t_c, Pt_c, label="20-min compartment")
plt.xlabel("Time [min]")
plt.ylabel("Pressure [bar, simplified]")
plt.title("A hidden compartment state following dive history")
plt.grid(True)
plt.legend()
plt.show()

The compartment lags behind the driving pressure.

That lag is the memory of the previous exposure.

# 12. Multiple compartments

For $n$ compartments:

$$
\dot P_{t,i}
=
k_i(P_a-P_{t,i}).
$$

Each has its own half-time.

In [ ]:
half_times = np.array([5, 10, 20, 40, 80], dtype=float)

def simulate_multiple_compartments(
    depth_fn,
    half_times,
    duration_min,
    dt_min=0.01,
    inert_fraction=0.79
):
    t = np.arange(0, duration_min + dt_min, dt_min)
    n_comp = len(half_times)

    depth = np.zeros_like(t)
    Pa = np.zeros_like(t)
    Pt = np.zeros((len(t), n_comp))

    Pt[0, :] = inert_gas_pressure(0, inert_fraction)
    k = np.log(2)/half_times

    for i in range(len(t)-1):
        depth[i] = depth_fn(t[i])
        Pa[i] = inert_gas_pressure(depth[i], inert_fraction)
        Pt[i+1] = Pt[i] + k*(Pa[i]-Pt[i])*dt_min

    depth[-1] = depth_fn(t[-1])
    Pa[-1] = inert_gas_pressure(depth[-1], inert_fraction)

    return t, depth, Pa, Pt

In [ ]:
t_m, z_m, Pa_m, Pt_m = simulate_multiple_compartments(
    profile,
    half_times,
    duration_min=50
)

for j, T_half in enumerate(half_times):
    plt.plot(t_m, Pt_m[:, j], label=f"{T_half:.0f} min")

plt.plot(t_m, Pa_m, linestyle="--", label="Driving pressure")
plt.xlabel("Time [min]")
plt.ylabel("Pressure [bar, simplified]")
plt.title("Multiple compartments")
plt.grid(True)
plt.legend()
plt.show()

Fast compartments follow recent changes more strongly.

Slow compartments preserve more of the earlier history.

# 13. State-space representation

Define:

$$
x=
\begin{bmatrix}
P_{t,1}\\
P_{t,2}\\
\vdots\\
P_{t,n}
\end{bmatrix}.
$$

Then:

$$
\dot x=Ax+BP_a(t)
$$

with:

$$
A=-\operatorname{diag}(k_i)
$$

and:

$$
B=
\begin{bmatrix}
k_1\\
k_2\\
\vdots
\end{bmatrix}.
$$

In [ ]:
k_vec = np.log(2)/half_times

A = -np.diag(k_vec)
B = k_vec.reshape(-1, 1)

print("A =")
print(A)
print()
print("B =")
print(B)

# 14. Eigenvalues and half-times

The eigenvalues are:

$$
\boxed{
\lambda_i=-k_i
}
$$

so every compartment is stable.

In [ ]:
print("Eigenvalues:")
print(np.linalg.eigvals(A))

This connects:

$$
\boxed{
T_{1/2}
\rightarrow
k_i
\rightarrow
\lambda_i
\rightarrow
e^{-k_i t}
}
$$

# 15. Transfer-function representation

For one compartment:

$$
\dot P_t+kP_t=kP_a.
$$

Taking Laplace transforms:

$$
(s+k)P_t(s)=kP_a(s).
$$

Therefore:

$$
\boxed{
\frac{P_t(s)}{P_a(s)}
=
\frac{k}{s+k}
}
$$

Each compartment is a first-order low-pass filter.

Fast compartments have larger bandwidth.

Slow compartments have lower bandwidth.

So a multi-compartment model acts like a **filter bank over the history of ambient pressure**.

# 16. Hidden-state interpretation

The pressure sensor does not measure:

$$
P_{t,1},\ldots,P_{t,n}.
$$

The dive computer computes these states from:

- depth history;
- gas assumptions;
- the model equations.

So compartment tensions are **model states**, not sensor readings.

# 17. Post-dive elimination

Extend the simulation after surfacing.

In [ ]:
t_long, z_long, Pa_long, Pt_long = simulate_multiple_compartments(
    profile,
    half_times,
    duration_min=180
)

for j, T_half in enumerate(half_times):
    plt.plot(t_long, Pt_long[:, j], label=f"{T_half:.0f} min")

plt.plot(t_long, Pa_long, linestyle="--", label="Driving pressure")
plt.xlabel("Time [min]")
plt.ylabel("Pressure [bar, simplified]")
plt.title("Post-profile elimination")
plt.grid(True)
plt.legend()
plt.show()

Slow compartments retain a longer memory of the exposure.

# 18. Same depth, different history

Two divers can be at the same current depth but have different compartment vectors because their previous profiles differ.

Therefore:

$$
\boxed{
\text{same depth now}
\not\Rightarrow
\text{same hidden state}
}
$$

This is one of the central ideas of decompression modeling.

# 19. A compartment is not an anatomical tissue

A mathematical compartment represents a characteristic response time.

It is an abstraction.

The model is not directly measuring a particular anatomical tissue.

# 20. Threshold logic — concept only

A decompression algorithm needs rules that interpret compartment states relative to ambient pressure.

Conceptually:

$$
\text{constraint}
=
f(P_{t,i},P_{\text{amb}}).
$$

Different algorithms use different structures and parameters.

We deliberately do **not** implement operational decompression limits or schedules here.

# 21. Numerical integration and real-time computation

A model-based dive computer can update compartments repeatedly:

1. measure pressure;
2. estimate depth;
3. compute driving inert-gas pressure;
4. update every compartment;
5. store the new hidden states.

This is real-time numerical integration.

# 22. Integration-step experiment

In [ ]:
for dt_test in [0.01, 0.1, 0.5]:
    t_test, z_test, Pa_test, Pt_test = simulate_compartment(
        profile,
        half_time_min=20,
        duration_min=50,
        dt_min=dt_test
    )

    plt.plot(t_test, Pt_test, label=f"dt={dt_test} min")

plt.xlabel("Time [min]")
plt.ylabel("Compartment tension")
plt.title("Effect of integration step")
plt.grid(True)
plt.legend()
plt.show()

# 23. Connection to Notebook 17

Each compartment has:

$$
G_i(s)=\frac{k_i}{s+k_i}.
$$

So the decompression model contains multiple first-order low-pass responses with different corner frequencies.

This is a frequency-domain view of physiological time scales.

# 24. Connection to Notebook 20

Notebook 20 gave the architecture:

$$
\text{pressure}
\rightarrow
\text{depth}
\rightarrow
\text{history}
\rightarrow
\text{model}.
$$

Now the model contains hidden states:

$$
P_{t,1},\ldots,P_{t,n}.
$$

# 25. The full state is growing

Across DiveLab we have seen states such as:

$$
[z,v]
$$

$$
[z,v,V_s]
$$

$$
[z,v,G]
$$

and now:

$$
\boxed{
x=
[z,v,G,P_{t,1},\ldots,P_{t,n}]^T
}
$$

The overall diving system contains many interacting time scales.

# 26. Multiple time scales

Examples:

### Fast

- vertical motion;
- breathing dynamics.

### Intermediate

- buoyancy adjustment;
- gas consumption.

### Slow

- some compartment states.

This kind of multi-timescale behavior appears throughout engineering and biology.

# 27. Important limitations

This notebook intentionally does not implement:

- decompression ceilings;
- operational stop schedules;
- gradient factors;
- real algorithm coefficient sets;
- mixed-gas planning;
- repetitive-dive rules.

It teaches the dynamical structure only.

# Exercises

### 1. Half-time conversion

Compute $k$ for:

$$
5,\ 10,\ 30,\ 60,\ 120\ \text{min}.
$$

In [ ]:
# Your code here

### 2. Verify one half-time

For a 20-minute compartment, simulate a constant driving pressure.

Check that after 20 minutes, half the original difference remains.

In [ ]:
# Your code here

### 3. Elimination

Start with elevated compartment tension and return the driving pressure to surface level.

Compare fast and slow compartments.

In [ ]:
# Your code here

### 4. Different histories

Construct two depth profiles with the same final depth and time but different earlier exposure.

Compare the compartment vectors.

In [ ]:
# Your code here

### 5. State-space model

Build $A$ and $B$ for your own compartment half-times.

Compute the eigenvalues and relate them to half-times.

In [ ]:
# Your code here

# Challenge — model-based dive-computer state engine

Combine:

- noisy pressure measurement;
- filtered depth;
- gas remaining;
- multiple compartment states.

Build a combined hidden state such as:

$$
x=
[z,G,P_{t,1},\ldots,P_{t,n}]^T.
$$

Drive it with a depth profile and plot all internal states.

Do not add operational decompression decision rules: focus on the state engine itself.

In [ ]:
# Your code here

# Summary

A simplified compartment follows:

$$
\boxed{
\dot P_t=k(P_a-P_t)
}
$$

with:

$$
\boxed{
k=\frac{\ln2}{T_{1/2}}
}
$$

and constant-input response:

$$
P_t(t)
=
P_a+
(P_t(0)-P_a)e^{-kt}.
$$

We learned that:

- compartments are first-order dynamical systems;
- half-time determines response speed;
- uptake and elimination use the same equation;
- multiple compartments store different time scales of history;
- compartment states are hidden model states;
- the model has a state-space representation;
- each compartment is a low-pass filter;
- current depth alone does not determine internal model state.

### Core insight

$$
\boxed{
\text{depth history}
\rightarrow
\text{hidden compartment states}
}
$$

and:

$$
\boxed{
\text{same depth now}
\not\Rightarrow
\text{same internal state}
}
$$

This completes the main topics in the original DiveLab roadmap.

### Next

We should now reorganize Notebooks 01–21 into the intended Modules I–IV, identify overlaps and gaps, and update `CURRICULUM.md` and `ROADMAP.md`.